# Fine-tune Qwen2.5-7B (QLoRA + Unsloth) — penalar operasi-form time series

Notebook Run-All untuk melatih model penalar yang mengeluarkan reasoning **operasi-form**
(`LANGKAH n: operasi = hasil` → `JAWABAN: label`) atas deret waktu. Setiap angka reasoning
diverifikasi ulang oleh *verifier* deterministik; notebook ini hanya melatih struktur
penalarannya.

**Input:** `data/train_unsloth.jsonl` (1172 sampel: 572 deret **nyata BPS** + 600
sintetis-terkendali; format chat `messages`, semua 100% grounded, anti-bocor lawan test).

**Target:** LoRA adapter di atas `Qwen2.5-7B-Instruct` (QLoRA NF4 4-bit).

**Hardware:** cukup GPU 16 GB (Colab/Kaggle T4 gratis) — pilih runtime **GPU** dulu.

> **Keamanan:** notebook TIDAK menaruh token apa pun. Token HuggingFace (opsional, hanya
> bila mau push adapter) diminta via `getpass` saat runtime dan tak tersimpan.


## 1 · Environment
Install Unsloth + dependensi (~2-3 menit). `trl` dipin <0.9 agar API `SFTTrainer` + `train_on_responses_only` di notebook ini stabil.

In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes datasets


In [ ]:
import torch
assert torch.cuda.is_available(), "Pilih runtime GPU dulu (Colab: Runtime > Change runtime type > GPU)."
print("GPU:", torch.cuda.get_device_name(0))


## 2 · Konfigurasi
Ubah di sini bila perlu — sisa notebook memakai nilai ini.

In [ ]:
# --- sumber data (repo publik proyek) ---
REPO_URL   = "https://github.com/cliprompter/creditaudit.git"  # ganti bila repo pindah
DATA_PATH  = "creditaudit/data/train_unsloth.jsonl"            # hasil git clone di bawah

# --- model & QLoRA ---
BASE_MODEL     = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # pra-kuantisasi 4-bit, unduh cepat, tak gated
MAX_SEQ_LEN    = 2048
LORA_R         = 32       # rank (16-64)
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.0

# --- training (hemat, muat di T4) ---
EPOCHS         = 2        # 2-3
LR             = 2e-4     # 1e-4 - 2e-4
BATCH_SIZE     = 2
GRAD_ACCUM     = 4        # batch efektif = 8
WARMUP_RATIO   = 0.05
SEED           = 3407
OUT_DIR        = "lora_qwen_penalar_ts"   # folder adapter hasil


## 3 · Ambil data train
Clone repo publik untuk mengambil `train_unsloth.jsonl`. Kalau lebih suka unggah manual, isi `DATA_PATH` ke file yang kamu upload.

In [ ]:
import os
from datasets import load_dataset

if not os.path.exists(DATA_PATH):
    !git clone --depth 1 $REPO_URL
assert os.path.exists(DATA_PATH), f"{DATA_PATH} tak ada — cek REPO_URL atau upload manual."

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print("sampel:", len(dataset))
print("contoh peran:", [m["role"] for m in dataset[0]["messages"]])
print("asisten[0][:200]:\n", dataset[0]["messages"][2]["content"][:200])


## 4 · Muat model (QLoRA 4-bit)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype         = None,        # auto: bf16 bila didukung, else fp16
    load_in_4bit  = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",   # hemat VRAM
    random_state   = SEED,
)


## 5 · Format chat (template Qwen-2.5)
Data sudah `messages` (system/user/assistant). Terapkan chat template Qwen-2.5 → kolom `text` untuk trainer.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def _format(batch):
    return {"text": [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch["messages"]
    ]}

dataset = dataset.map(_format, batched=True, remove_columns=dataset.column_names)
print(dataset[0]["text"][:400])


## 6 · Trainer (SFT, hitung loss hanya pada jawaban)
`train_on_responses_only` menutup bagian prompt → model belajar **mengeluarkan** reasoning, bukan menghafal soal.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = dataset,
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LEN,
    dataset_num_proc = 2,
    packing          = False,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio     = WARMUP_RATIO,
        num_train_epochs = EPOCHS,
        learning_rate    = LR,
        lr_scheduler_type = "cosine",
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps    = 10,
        optim            = "adamw_8bit",
        weight_decay     = 0.01,
        seed             = SEED,
        output_dir       = "outputs",
        report_to        = "none",
    ),
)

# Mask prompt: loss hanya pada bagian asisten (Qwen-2.5 markers).
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)


## 7 · Latih

In [ ]:
stats = trainer.train()
print("waktu latih (detik):", stats.metrics.get("train_runtime"))
print("loss akhir:", stats.metrics.get("train_loss"))


## 8 · Simpan adapter LoRA
Tersimpan lokal. Push ke HuggingFace **opsional** (butuh token write).

In [ ]:
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("adapter tersimpan di:", OUT_DIR)

# --- opsional: push ke HuggingFace Hub ---
# import getpass
# from huggingface_hub import login
# tok = getpass.getpass("HF token (write) — Enter kosong untuk skip: ")
# if tok:
#     login(token=tok)
#     REPO = "username/qwen-penalar-ts-lora"   # ganti username
#     model.push_to_hub(REPO, token=tok)
#     tokenizer.push_to_hub(REPO, token=tok)
#     print("ter-push ke", REPO)


## 9 · Uji cepat (sanity generate)
Jalankan model pada satu deret contoh — cek keluaran berformat `LANGKAH … JAWABAN:`.

In [ ]:
FastLanguageModel.for_inference(model)

# ambil prompt (system+user) dari satu sampel mentah
raw = load_dataset("json", data_files=DATA_PATH, split="train")
msgs = raw[0]["messages"][:2]   # system + user saja (tanpa jawaban gold)

inputs = tokenizer.apply_chat_template(
    msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=256, do_sample=False,
                     use_cache=True, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))


---
### Langkah berikut (di luar notebook)
Evaluasi adapter ini dengan **harness verifier** proyek (grounding score + akurasi + efisiensi token)
lawan `data/processed/benchmark_uji.jsonl` untuk RQ3 (vs baseline tanpa fine-tune). Adapter di `OUT_DIR`
tinggal dimuat ulang lewat `FastLanguageModel.from_pretrained(OUT_DIR)`.
